In [0]:

# STEP 1: CREATE GOLD SCHEMA

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold
""")

print("Gold schema created successfully!")


Gold schema created successfully!


In [0]:

# STEP 2: LOAD SILVER TABLES

customers = spark.table("workspace.silver.customers_clean")

orders = spark.table("workspace.silver.orders_clean")

items = spark.table("workspace.silver.order_items_clean")

payments = spark.table("workspace.silver.payments_clean")

products = spark.table("workspace.silver.products_clean")

reviews = spark.table("workspace.silver.reviews_clean")

sellers = spark.table("workspace.silver.sellers_clean")

geo = spark.table("workspace.silver.geolocation_clean")

category = spark.table("workspace.silver.category_translation_clean")

print("All 9 Silver datasets loaded successfully!")


All 9 Silver datasets loaded successfully!


In [0]:

# STEP 3: VERIFY SILVER DATASETS

datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": items,
    "Payments": payments,
    "Products": products,
    "Reviews": reviews,
    "Sellers": sellers,
    "Geolocation": geo,
    "Category Translation": category
}

for name, df in datasets.items():

    print(name.upper())

    print("Total Records:", df.count())

    print("Columns:", df.columns)

    df.printSchema()


CUSTOMERS
Total Records: 99441
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'ingestion_timestamp', 'source_file']
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

ORDERS
Total Records: 99441
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'ingestion_timestamp', 'source_file']
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_

In [0]:

# GOLD TABLE 1: CUSTOMER BEHAVIOUR ANALYTICS
# STEP 1: LOAD REQUIRED DATASETS

from pyspark.sql.functions import (
    col, sum, countDistinct, first,
    min, max, when, round, coalesce
)

customers = spark.table(
    "workspace.silver.customers_clean"
)

orders = spark.table(
    "workspace.silver.orders_clean"
)

payments = spark.table(
    "workspace.silver.payments_clean"
)

print("Customer, Orders and Payments loaded successfully!")

print("Customers:", customers.count())
print("Orders:", orders.count())
print("Payments:", payments.count())


Customer, Orders and Payments loaded successfully!
Customers: 99441
Orders: 99441
Payments: 103886


In [0]:

# STEP 2: AGGREGATE PAYMENTS BY ORDER

order_payments = (
    payments
    .groupBy("order_id")
    .agg(
        sum("payment_value").alias("total_order_payment")
    )
)

print("Payment aggregation completed!")

print("Unique orders with payments:",
      order_payments.count())

display(order_payments.limit(10))


Payment aggregation completed!
Unique orders with payments: 99440


order_id,total_order_payment
a9810da82917af2d9aefd1278f1dcfa0,24.39
e8977f709bbb0367f51b589e6325f991,132.51
2e2c60b99754ae1e4d8b18846cfec9f2,542.66
60e43c5f5ba559d9dd19357ed72cb2f2,78.3
110d9355d6338eab0a65a0cacdde5434,566.4
0f4b5614da144989e2acc5291af24357,96.98
e2cd740333fe4585ead5c50c61edb6a5,163.25
14c0c0bebe2cd50fb1dbdee84fbae03a,180.55
55c0b6b41ce482c3f77407ef43fc6bad,26.08
c1b770015e27aaf2b475026989d04fe0,137.73


In [0]:

# STEP 3: JOIN CUSTOMER, ORDER AND PAYMENT DATA

# Select completed orders
delivered_orders = orders.filter(
    col("order_status") == "delivered"
)

# Join orders with customers
customer_orders = delivered_orders.join(
    customers.select(
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ),
    on="customer_id",
    how="inner"
)

# Join aggregated payments
customer_orders = customer_orders.join(
    order_payments,
    on="order_id",
    how="left"
)

print("Customer-order-payment join completed!")

print("Total joined records:", customer_orders.count())

display(customer_orders.limit(10))


Customer-order-payment join completed!
Total joined records: 96478


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,source_file,customer_unique_id,customer_city,customer_state,total_order_payment
eac4ffbe456464bdc3ce5f001b6439c5,237098a64674ae89babdc426746260fc,delivered,2017-04-14T11:24:56.000Z,2017-04-14T11:35:23.000Z,2017-04-17T10:43:50.000Z,2017-04-20T17:04:51.000Z,2017-05-10T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,4390ddbb6276a66ff1736a6710205dca,curitiba,PR,94.63
0330631bdb2ec8c341f487ed9c22fc8b,e3109970a3fe8021d5ff82c577ce5606,delivered,2018-05-25T12:02:08.000Z,2018-05-26T02:31:54.000Z,2018-05-29T15:41:00.000Z,2018-06-07T22:42:02.000Z,2018-06-26T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,a8654e2af5da6bb72f52c22b164855e1,sao paulo,SP,58.69
04a92860751dfe7ae41f71e1228ffd18,c532a74a3ebf1bacce2e2bcce3783317,delivered,2018-05-14T13:30:21.000Z,2018-05-14T13:56:47.000Z,2018-05-24T14:26:00.000Z,2018-06-04T20:48:29.000Z,2018-06-12T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,91ec50a00ae74d0a229d2efdf4344e1e,ribeirao preto,SP,218.49
14282bc70be9bdda515182fb1ce62af4,19cecb194f54e614b70d971306a9931b,delivered,2018-04-18T14:18:09.000Z,2018-04-19T02:52:02.000Z,2018-04-20T00:47:44.000Z,2018-04-26T16:26:38.000Z,2018-05-11T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,d251c190ca75786e9ab937982d60d1d4,belo horizonte,MG,50.92
89e954a7919e964597b60e9f016df7fd,c82a5e4fafdbeb34f08928ccfba27d14,delivered,2017-05-15T23:34:21.000Z,2017-05-16T19:15:11.000Z,2017-05-18T11:37:59.000Z,2017-05-25T12:39:52.000Z,2017-06-08T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,ca19a17e381182923b66007a351574b7,foz do iguacu,PR,47.74
6fa2b34b44af3aa81c0b080d0c133338,b06429ef920fcfdd75713c712c9ee7b7,delivered,2018-06-04T15:30:50.000Z,2018-06-04T15:52:08.000Z,2018-06-06T15:26:00.000Z,2018-06-07T21:34:00.000Z,2018-06-28T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,9316f45a5da8403a5938bd6069b1a4a7,sao jose dos campos,SP,78.28
d209d8b7e741f8ce447b0173e579e3a4,d3ab15f0bd2c58865d566ab645572cd5,delivered,2018-02-01T19:34:28.000Z,2018-02-01T19:45:25.000Z,2018-02-05T13:53:06.000Z,2018-02-06T19:03:16.000Z,2018-02-20T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,9ccfff93c79f3dd996cce15f26480c5b,rio de janeiro,RJ,72.38
dd621d7d942344da783d1d390dc06f17,031cd5f826be3d804771e3e3a1b21a1c,delivered,2017-04-30T20:04:38.000Z,2017-05-03T14:25:41.000Z,2017-05-04T15:32:31.000Z,2017-05-23T21:02:34.000Z,2017-06-07T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,717aa48025662fcf27ddebbecc5f782b,salvador,BA,147.17
72ac9b27dfe6809898729ee40dd3e353,79adcf02229a33e78f0f5412a2434f53,delivered,2018-05-02T21:45:22.000Z,2018-05-02T22:34:48.000Z,2018-05-04T12:10:00.000Z,2018-05-11T18:37:39.000Z,2018-05-30T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,8c8fccc50566baaed602e3775d9d5665,rio de janeiro,RJ,114.27
8d9c47d02bd99ae8d7dd151ededa9281,e3c7e245a96d7fa339fe6c16f8da4e90,delivered,2017-10-15T17:12:19.000Z,2017-10-15T17:56:24.000Z,2017-10-17T21:05:06.000Z,2017-10-23T22:04:54.000Z,2017-11-01T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,79051ee5ee98c4bd6982e67e2e79dbcb,franco da rocha,SP,415.97


In [0]:

# STEP 4: CUSTOMER BEHAVIOUR AGGREGATION

gold_customer = (
    customer_orders
    .groupBy("customer_unique_id")
    .agg(

        # Total orders placed
        countDistinct("order_id").alias("total_orders"),

        # Total money spent on delivered orders
        sum("total_order_payment").alias("total_spending"),

        # First purchase date
        min("order_purchase_timestamp").alias("first_purchase_date"),

        # Most recent purchase date
        max("order_purchase_timestamp").alias("last_purchase_date"),

        # Customer location
        first("customer_city", ignorenulls=True).alias("customer_city"),

        first("customer_state", ignorenulls=True).alias("customer_state")
    )
)

# Calculate average order value
gold_customer = gold_customer.withColumn(
    "average_order_value",
    round(
        col("total_spending") / col("total_orders"),
        2
    )
)

# Identify repeat customers
gold_customer = gold_customer.withColumn(
    "customer_type",
    when(col("total_orders") > 1, "Repeat Customer")
    .otherwise("One-time Customer")
)

print("Customer behaviour calculations completed!")

display(gold_customer.limit(10))


Customer behaviour calculations completed!


customer_unique_id,total_orders,total_spending,first_purchase_date,last_purchase_date,customer_city,customer_state,average_order_value,customer_type
6d8a1ff2b74ef9d939013151ef4d95cf,1,114.77000000000001,2017-04-22T23:14:00.000Z,2017-04-22T23:14:00.000Z,rio de janeiro,RJ,114.77,One-time Customer
55507ccf4995e1ed3ef6e49ea7c27f8a,1,55.29,2018-05-11T10:05:25.000Z,2018-05-11T10:05:25.000Z,chavantes,SP,55.29,One-time Customer
a80a15cc7984d350d36af32d10866143,1,212.88,2018-01-16T11:27:24.000Z,2018-01-16T11:27:24.000Z,sao paulo,SP,212.88,One-time Customer
e7448e1e1479af466988b0b5a3f7f565,1,175.42,2018-08-12T11:56:55.000Z,2018-08-12T11:56:55.000Z,sao paulo,SP,175.42,One-time Customer
dc3c039d7e42558e1d230dafba00d888,1,60.72,2018-02-18T21:30:04.000Z,2018-02-18T21:30:04.000Z,barueri,SP,60.72,One-time Customer
8cdbbdb32c682663f17135569a85c24f,1,196.97,2017-09-30T15:00:21.000Z,2017-09-30T15:00:21.000Z,sao paulo,SP,196.97,One-time Customer
449c102b7926fd08fbfcbb387a6e404f,1,72.53,2018-01-21T20:33:38.000Z,2018-01-21T20:33:38.000Z,tubarao,SC,72.53,One-time Customer
922106d9e4ab35ddf6a761c6137ab813,1,322.68,2017-04-19T17:26:36.000Z,2017-04-19T17:26:36.000Z,rio de janeiro,RJ,322.68,One-time Customer
d8cd82807a4410f5091fa7e41b0dad0d,1,336.94,2017-08-21T01:18:30.000Z,2017-08-21T01:18:30.000Z,valparaiso de goias,GO,336.94,One-time Customer
b0bd994fd9d1c590949928d2f319587f,1,127.08,2018-08-15T17:00:09.000Z,2018-08-15T17:00:09.000Z,campinas,SP,127.08,One-time Customer


In [0]:

# STEP 5: SAVE GOLD CUSTOMER ANALYTICS

(gold_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_customer_analytics")
)

print("Gold Customer Analytics table created successfully!")


Gold Customer Analytics table created successfully!


In [0]:

# STEP 6: VALIDATE GOLD CUSTOMER TABLE

gold_customer_check = spark.table(
    "workspace.gold.gold_customer_analytics"
)

print("GOLD CUSTOMER ANALYTICS")
print("-------------------------------")

print("Total customers:",
      gold_customer_check.count())

print("Columns:",
      gold_customer_check.columns)

gold_customer_check.printSchema()

display(gold_customer_check.limit(20))


GOLD CUSTOMER ANALYTICS
-------------------------------
Total customers: 93358
Columns: ['customer_unique_id', 'total_orders', 'total_spending', 'first_purchase_date', 'last_purchase_date', 'customer_city', 'customer_state', 'average_order_value', 'customer_type']
root
 |-- customer_unique_id: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_spending: double (nullable = true)
 |-- first_purchase_date: timestamp (nullable = true)
 |-- last_purchase_date: timestamp (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- average_order_value: double (nullable = true)
 |-- customer_type: string (nullable = true)



customer_unique_id,total_orders,total_spending,first_purchase_date,last_purchase_date,customer_city,customer_state,average_order_value,customer_type
6d8a1ff2b74ef9d939013151ef4d95cf,1,114.77000000000001,2017-04-22T23:14:00.000Z,2017-04-22T23:14:00.000Z,rio de janeiro,RJ,114.77,One-time Customer
55507ccf4995e1ed3ef6e49ea7c27f8a,1,55.29,2018-05-11T10:05:25.000Z,2018-05-11T10:05:25.000Z,chavantes,SP,55.29,One-time Customer
a80a15cc7984d350d36af32d10866143,1,212.88,2018-01-16T11:27:24.000Z,2018-01-16T11:27:24.000Z,sao paulo,SP,212.88,One-time Customer
e7448e1e1479af466988b0b5a3f7f565,1,175.42,2018-08-12T11:56:55.000Z,2018-08-12T11:56:55.000Z,sao paulo,SP,175.42,One-time Customer
dc3c039d7e42558e1d230dafba00d888,1,60.72,2018-02-18T21:30:04.000Z,2018-02-18T21:30:04.000Z,barueri,SP,60.72,One-time Customer
8cdbbdb32c682663f17135569a85c24f,1,196.97,2017-09-30T15:00:21.000Z,2017-09-30T15:00:21.000Z,sao paulo,SP,196.97,One-time Customer
449c102b7926fd08fbfcbb387a6e404f,1,72.53,2018-01-21T20:33:38.000Z,2018-01-21T20:33:38.000Z,tubarao,SC,72.53,One-time Customer
922106d9e4ab35ddf6a761c6137ab813,1,322.68,2017-04-19T17:26:36.000Z,2017-04-19T17:26:36.000Z,rio de janeiro,RJ,322.68,One-time Customer
d8cd82807a4410f5091fa7e41b0dad0d,1,336.94,2017-08-21T01:18:30.000Z,2017-08-21T01:18:30.000Z,valparaiso de goias,GO,336.94,One-time Customer
b0bd994fd9d1c590949928d2f319587f,1,127.08,2018-08-15T17:00:09.000Z,2018-08-15T17:00:09.000Z,campinas,SP,127.08,One-time Customer


In [0]:

# GOLD TABLE 2: SALES PERFORMANCE ANALYTICS
# STEP 1: LOAD DATA

from pyspark.sql.functions import (
    col, sum, countDistinct, when,
    date_format, round, coalesce, lit
)

orders = spark.table(
    "workspace.silver.orders_clean"
)

payments = spark.table(
    "workspace.silver.payments_clean"
)

print("Orders:", orders.count())
print("Payments:", payments.count())


Orders: 99441
Payments: 103886


In [0]:

# STEP 2: AGGREGATE ORDER PAYMENTS

order_payments = (
    payments
    .groupBy("order_id")
    .agg(
        sum("payment_value").alias("order_revenue")
    )
)

print("Payment aggregation completed!")

display(order_payments.limit(10))


Payment aggregation completed!


order_id,order_revenue
a9810da82917af2d9aefd1278f1dcfa0,24.39
e8977f709bbb0367f51b589e6325f991,132.51
2e2c60b99754ae1e4d8b18846cfec9f2,542.66
60e43c5f5ba559d9dd19357ed72cb2f2,78.3
110d9355d6338eab0a65a0cacdde5434,566.4
0f4b5614da144989e2acc5291af24357,96.98
e2cd740333fe4585ead5c50c61edb6a5,163.25
14c0c0bebe2cd50fb1dbdee84fbae03a,180.55
55c0b6b41ce482c3f77407ef43fc6bad,26.08
c1b770015e27aaf2b475026989d04fe0,137.73


In [0]:

# STEP 3: JOIN ORDERS AND PAYMENTS

sales_base = (
    orders
    .join(
        order_payments,
        on="order_id",
        how="left"
    )
)

print("Orders and Payments joined successfully!")

print("Total records:", sales_base.count())

display(sales_base.limit(10))


Orders and Payments joined successfully!
Total records: 99441


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,source_file,order_revenue
2807d0e504d6d4894d41672727bc139f,72ae281627a6102d9b3718528b420f8a,delivered,2018-02-03T20:37:35.000Z,2018-02-03T20:50:22.000Z,2018-02-05T22:37:28.000Z,2018-02-08T16:13:46.000Z,2018-02-21T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,17.28
ccbabeb0b02433bd0fcbac46e70339f2,c77ee2d8ba1614a4d489a44166894938,delivered,2018-02-19T20:31:09.000Z,2018-02-21T06:15:25.000Z,2018-02-22T21:04:23.000Z,2018-03-09T22:22:25.000Z,2018-03-13T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,43.0
e4de6d53ecff736bc68804b0b6e9f635,9f6618c17568ac301465fe7ad056c674,delivered,2017-10-16T14:56:50.000Z,2017-10-17T03:49:34.000Z,2017-10-27T22:14:21.000Z,2017-11-08T21:25:24.000Z,2017-11-21T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,231.12
cadbb3657dac2dbbd5b84b12e7b78aad,93ada7a24817edda9f4ab998fa823d16,delivered,2018-02-27T12:55:42.000Z,2018-03-01T02:48:54.000Z,2018-03-03T02:27:03.000Z,2018-03-16T14:59:01.000Z,2018-03-29T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,409.79
d3d6788577c9592da441752e8a1dd5e3,8628fac2267e8c8804525da99c10ed0e,delivered,2017-09-19T22:17:15.000Z,2017-09-20T07:55:14.000Z,2017-09-22T17:23:09.000Z,2017-10-10T18:43:53.000Z,2017-10-13T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,76.65
6a0a8bfbbe700284feb0845d95e0867f,68451b39b1314302c08c65a29f1140fc,delivered,2017-11-22T11:32:22.000Z,2017-11-22T11:46:50.000Z,2017-11-27T13:39:35.000Z,2017-12-28T19:43:00.000Z,2017-12-11T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,101.74
0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03T17:44:42.000Z,2018-08-07T06:15:14.000Z,null,null,2018-08-21T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,50.35
e1da8361c76cab67aa3588a1fbf1af54,dd854e24b40e3bc2b306946dee252015,delivered,2017-07-01T15:42:32.000Z,2017-07-04T04:10:19.000Z,2017-07-04T14:52:46.000Z,2017-07-12T20:17:57.000Z,2017-07-25T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,253.4
4b3a605942f29d490cb74bd6ace6b9f0,e8a332c3433fbd37920ec89fc295d075,delivered,2017-09-29T09:32:17.000Z,2017-09-29T09:44:12.000Z,2017-09-29T21:58:48.000Z,2017-09-30T16:55:39.000Z,2017-10-17T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,38.1
741fd1b80f015f170a7f458cf86769b8,df5c9e02596851fcbfc98804e4108bbd,delivered,2017-12-02T14:28:03.000Z,2017-12-02T14:38:30.000Z,2017-12-04T21:54:27.000Z,2017-12-11T16:17:04.000Z,2018-01-04T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv,106.38


In [0]:

# STEP 4: MONTHLY SALES ANALYTICS

monthly_sales = (
    sales_base
    .filter(col("order_purchase_timestamp").isNotNull())
    .withColumn(
        "sales_month",
        date_format(
            col("order_purchase_timestamp"),
            "yyyy-MM"
        )
    )
    .groupBy("sales_month")
    .agg(

        # Total orders
        countDistinct("order_id").alias("total_orders"),

        # Delivered orders
        countDistinct(
            when(
                col("order_status") == "delivered",
                col("order_id")
            )
        ).alias("delivered_orders"),

        # Cancelled orders
        countDistinct(
            when(
                col("order_status") == "canceled",
                col("order_id")
            )
        ).alias("cancelled_orders"),

        # Revenue from delivered orders only
        sum(
            when(
                col("order_status") == "delivered",
                coalesce(col("order_revenue"), lit(0.0))
            ).otherwise(0.0)
        ).alias("total_revenue")
    )
)

# Calculate average order value
monthly_sales = monthly_sales.withColumn(
    "average_order_value",
    round(
        col("total_revenue") /
        when(col("delivered_orders") > 0,
             col("delivered_orders")),
        2
    )
)

print("Monthly sales analytics completed!")

display(monthly_sales.orderBy("sales_month"))


Monthly sales analytics completed!


sales_month,total_orders,delivered_orders,cancelled_orders,total_revenue,average_order_value
2016-09,4,1,2,0.0,0.0
2016-10,324,265,24,46566.70999999999,175.72
2016-12,1,1,0,19.62,19.62
2017-01,800,750,3,127545.66999999993,170.06
2017-02,1780,1653,17,271298.64999999944,164.13
2017-03,2682,2546,33,414369.3900000014,162.75
2017-04,2404,2303,18,390952.1799999996,169.76
2017-05,3700,3546,29,567066.7300000029,159.92
2017-06,3245,3135,16,490225.6000000026,156.37
2017-07,4026,3872,28,566403.9300000041,146.28


In [0]:

# STEP 5: SAVE GOLD SALES TABLE

(monthly_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_sales_analytics")
)

print("Gold Sales Analytics table created successfully!")


Gold Sales Analytics table created successfully!


In [0]:

# STEP 6: VALIDATE SALES TABLE

gold_sales_check = spark.table(
    "workspace.gold.gold_sales_analytics"
)

print("GOLD SALES ANALYTICS")
print("----------------------------")

print("Total monthly records:",
      gold_sales_check.count())

gold_sales_check.printSchema()

display(
    gold_sales_check.orderBy("sales_month")
)


GOLD SALES ANALYTICS
----------------------------
Total monthly records: 25
root
 |-- sales_month: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- delivered_orders: long (nullable = true)
 |-- cancelled_orders: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- average_order_value: double (nullable = true)



sales_month,total_orders,delivered_orders,cancelled_orders,total_revenue,average_order_value
2016-09,4,1,2,0.0,0.0
2016-10,324,265,24,46566.70999999999,175.72
2016-12,1,1,0,19.62,19.62
2017-01,800,750,3,127545.66999999993,170.06
2017-02,1780,1653,17,271298.64999999944,164.13
2017-03,2682,2546,33,414369.3900000014,162.75
2017-04,2404,2303,18,390952.1799999996,169.76
2017-05,3700,3546,29,567066.7300000029,159.92
2017-06,3245,3135,16,490225.6000000026,156.37
2017-07,4026,3872,28,566403.9300000041,146.28


In [0]:

from pyspark.sql.functions import round, col

sales_rounded = (
    spark.table("workspace.gold.gold_sales_analytics")
    .withColumn("total_revenue", round(col("total_revenue"), 2))
    .withColumn("average_order_value", round(col("average_order_value"), 2))
)

(sales_rounded.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_sales_analytics")
)

print("Sales table formatting updated!")
display(spark.table("workspace.gold.gold_sales_analytics"))


Sales table formatting updated!


sales_month,total_orders,delivered_orders,cancelled_orders,total_revenue,average_order_value
2017-12,5673,5513,11,843199.17,152.95
2018-02,6728,6555,73,966510.88,147.45
2017-02,1780,1653,17,271298.65,164.13
2016-12,1,1,0,19.62,19.62
2018-07,6292,6159,41,1027903.86,166.89
2018-10,4,0,4,0.0,null
2018-06,6167,6099,18,1012090.68,165.94
2017-05,3700,3546,29,567066.73,159.92
2016-10,324,265,24,46566.71,175.72
2017-04,2404,2303,18,390952.18,169.76


In [0]:

# GOLD TABLE 3: PRODUCT DEMAND ANALYTICS
# STEP 1: LOAD SILVER TABLES

from pyspark.sql.functions import (
    col, sum, countDistinct, avg,
    round, desc
)

# Load Silver tables
products = spark.table(
    "workspace.silver.products_clean"
)

order_items = spark.table(
    "workspace.silver.order_items_clean"
)

orders = spark.table(
    "workspace.silver.orders_clean"
)

category_translation = spark.table(
    "workspace.silver.category_translation_clean"
)

print("PRODUCTS:", products.count())
print("ORDER ITEMS:", order_items.count())
print("ORDERS:", orders.count())
print("CATEGORY TRANSLATIONS:", category_translation.count())

print("\nPRODUCT SCHEMA")
products.printSchema()

print("\nORDER ITEMS SCHEMA")
order_items.printSchema()


PRODUCTS: 32951
ORDER ITEMS: 112650
ORDERS: 99441
CATEGORY TRANSLATIONS: 71

PRODUCT SCHEMA
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)


ORDER ITEMS SCHEMA
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_times

In [0]:

# STEP 2: JOIN PRODUCT SALES DATA

# Join order items with orders
product_sales = (
    order_items
    .join(
        orders.select("order_id", "order_status"),
        on="order_id",
        how="inner"
    )
)

# Keep only delivered orders
product_sales = product_sales.filter(
    col("order_status") == "delivered"
)

# Join with product information
product_sales = product_sales.join(
    products.select(
        "product_id",
        "product_category_name"
    ),
    on="product_id",
    how="left"
)

print("PRODUCT SALES DATA CREATED")
print("--------------------------------")

print("Total delivered product items:",
      product_sales.count())

print("Unique products:",
      product_sales.select("product_id").distinct().count())

print("Unique orders:",
      product_sales.select("order_id").distinct().count())

display(product_sales.limit(10))


PRODUCT SALES DATA CREATED
--------------------------------
Total delivered product items: 110197
Unique products: 32216
Unique orders: 96478


product_id,order_id,order_item_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp,source_file,order_status,product_category_name
5a419dbf24a8c9718fe522b81c69f61a,0010b2e5201cc5f1ae7e9c6cc8f5bd00,1,3504c0cb71d7fa48d967e0e4c94d59d9,2017-09-15T18:04:37.000Z,48.9,16.6,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,cool_stuff
059344baebbeaa42fa9f2bbe11b1583e,0028de0ca693a1bb26448916a81105cc,1,955fee9216a65b617aa5c0531780ce60,2018-08-21T03:35:17.000Z,29.99,15.31,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,construcao_ferramentas_iluminacao
08279c494018541f71443c07d77560f8,0032d07457ae9c806c79368d7d9ce96b,1,e333046ce6517bd8bb510291d44f0130,2018-03-15T19:28:51.000Z,159.0,27.19,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,utilidades_domesticas
9df0e8a7eef2a38b74e6d5c0e224b11f,00471463a6106056c1a2a809f70de640,1,7c67e1448b00f6e969d365cea6b010ab,2017-10-03T22:46:14.000Z,179.99,85.97,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,moveis_decoracao
fb7a100ec8c7b34f60cec22b1a9a10e0,005d9a5423d47281ac463a968b3936fb,1,d98eec89afa3380e14463da2aabaea72,2017-10-24T12:28:16.000Z,49.99,18.12,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,brinquedos
735413ccfa564132a231aad945ef85ad,0068e836900867da359bd81db9227a33,1,5b67882648febfb6e1225e6ad7164f6b,2017-10-05T02:24:43.000Z,96.99,16.44,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,esporte_lazer
aacfae7cd4bac4849766f640abf2db8a,006f7dfffe2d90809598e8f1972b829b,1,729b2d09b2a0bdab221076327f13d050,2018-03-28T23:07:23.000Z,39.85,12.79,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,beleza_saude
57f2bc497c1a3ebe41ba7a06d78ed159,008819c6d5f6da6fa5cd9d50ca927cf4,1,6560211a19b47992c3666cc44a7e94c0,2018-08-06T21:50:11.000Z,55.0,7.65,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,relogios_presentes
eb8dbe7d2dd22ab3547925e49a78feb3,00891ba5de66f55000ee358ceea9b345,1,218d46b86c1881d022bce9c68a7d4b15,2017-05-19T17:35:15.000Z,133.0,16.69,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,esporte_lazer
0ea04fc101523a27bb53845d14d7d809,00949655cdd8d0465e433a5e6c9643e5,1,8648b1e89e9b349e32d3741b30ec737e,2018-06-29T03:31:55.000Z,18.0,7.87,2026-11-16T06:16:18.770Z,olist_order_items_dataset.csv,delivered,construcao_ferramentas_construcao


In [0]:

# STEP 3: CATEGORY TRANSLATION

product_sales = product_sales.join(
    category_translation.select(
        "product_category_name",
        "product_category_name_english"
    ),
    on="product_category_name",
    how="left"
)

# Create an English category column
product_sales = product_sales.withColumn(
    "product_category",
    coalesce(
        col("product_category_name_english"),
        col("product_category_name")
    )
)

print("CATEGORY TRANSLATION COMPLETED")

display(
    product_sales.select(
        "product_id",
        "product_category",
        "price",
        "order_id"
    ).limit(10)
)


CATEGORY TRANSLATION COMPLETED


product_id,product_category,price,order_id
5a419dbf24a8c9718fe522b81c69f61a,cool_stuff,48.9,0010b2e5201cc5f1ae7e9c6cc8f5bd00
059344baebbeaa42fa9f2bbe11b1583e,construction_tools_lights,29.99,0028de0ca693a1bb26448916a81105cc
08279c494018541f71443c07d77560f8,housewares,159.0,0032d07457ae9c806c79368d7d9ce96b
9df0e8a7eef2a38b74e6d5c0e224b11f,furniture_decor,179.99,00471463a6106056c1a2a809f70de640
fb7a100ec8c7b34f60cec22b1a9a10e0,toys,49.99,005d9a5423d47281ac463a968b3936fb
735413ccfa564132a231aad945ef85ad,sports_leisure,96.99,0068e836900867da359bd81db9227a33
aacfae7cd4bac4849766f640abf2db8a,health_beauty,39.85,006f7dfffe2d90809598e8f1972b829b
57f2bc497c1a3ebe41ba7a06d78ed159,watches_gifts,55.0,008819c6d5f6da6fa5cd9d50ca927cf4
eb8dbe7d2dd22ab3547925e49a78feb3,sports_leisure,133.0,00891ba5de66f55000ee358ceea9b345
0ea04fc101523a27bb53845d14d7d809,construction_tools_construction,18.0,00949655cdd8d0465e433a5e6c9643e5


In [0]:

from pyspark.sql.functions import count, avg

print("Required functions imported successfully!")


Required functions imported successfully!


In [0]:

# STEP 4: PRODUCT PERFORMANCE AGGREGATION

product_analytics = (
    product_sales
    .groupBy(
        "product_id",
        "product_category"
    )
    .agg(

        # Total units sold
        count("order_item_id").alias("total_units_sold"),

        # Total unique orders
        countDistinct("order_id").alias("total_orders"),

        # Total product revenue
        sum("price").alias("total_revenue"),

        # Average selling price
        round(avg("price"), 2).alias("average_price"),

        # Total freight charges
        sum("freight_value").alias("total_freight")
    )
)

# Round revenue and freight
product_analytics = (
    product_analytics
    .withColumn(
        "total_revenue",
        round(col("total_revenue"), 2)
    )
    .withColumn(
        "total_freight",
        round(col("total_freight"), 2)
    )
)

print("PRODUCT PERFORMANCE ANALYTICS CREATED")
print("---------------------------------------")

print("Total products analyzed:",
      product_analytics.count())

display(
    product_analytics.orderBy(
        desc("total_revenue")
    ).limit(10)
)


PRODUCT PERFORMANCE ANALYTICS CREATED
---------------------------------------
Total products analyzed: 32216


product_id,product_category,total_units_sold,total_orders,total_revenue,average_price,total_freight
bb50f2e236e5eea0100680137654686c,health_beauty,194,186,63560.0,327.63,3698.03
6cdd53843498f92890544667809f1595,health_beauty,153,148,53652.3,350.67,4281.43
d6160fb7873f184099d9bc95e30376af,computers,33,33,45949.35,1392.4,1364.83
d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,332,313,45620.56,137.41,13336.75
99a4788cb24856965c36a24e339b6058,bed_bath_table,477,456,42049.66,88.15,7857.84
3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,272,253,40782.8,149.94,7093.26
25c38557cf793876c5abdd5931f922db,baby,38,38,38907.32,1023.88,1404.63
5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,63,63,37733.9,598.95,3991.91
53b36df67ebb7c41585e8d54d6772e08,watches_gifts,321,304,37454.63,116.68,2258.86
aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,520,425,37104.3,71.35,7093.45


In [0]:

# STEP 5: SAVE GOLD PRODUCT ANALYTICS

(product_analytics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_product_analytics")
)

print("Gold Product Analytics table created successfully!")


Gold Product Analytics table created successfully!


In [0]:

# STEP 6: VALIDATE GOLD PRODUCT TABLE

gold_product_check = spark.table(
    "workspace.gold.gold_product_analytics"
)

print("GOLD PRODUCT ANALYTICS")
print("--------------------------------")

print("Total products:",
      gold_product_check.count())

print("\nTable Schema:")
gold_product_check.printSchema()

print("\nTOP 10 PRODUCTS BY REVENUE")

display(
    gold_product_check
    .orderBy(desc("total_revenue"))
    .limit(10)
)


GOLD PRODUCT ANALYTICS
--------------------------------
Total products: 32216

Table Schema:
root
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- total_units_sold: long (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- average_price: double (nullable = true)
 |-- total_freight: double (nullable = true)


TOP 10 PRODUCTS BY REVENUE


product_id,product_category,total_units_sold,total_orders,total_revenue,average_price,total_freight
bb50f2e236e5eea0100680137654686c,health_beauty,194,186,63560.0,327.63,3698.03
6cdd53843498f92890544667809f1595,health_beauty,153,148,53652.3,350.67,4281.43
d6160fb7873f184099d9bc95e30376af,computers,33,33,45949.35,1392.4,1364.83
d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,332,313,45620.56,137.41,13336.75
99a4788cb24856965c36a24e339b6058,bed_bath_table,477,456,42049.66,88.15,7857.84
3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,272,253,40782.8,149.94,7093.26
25c38557cf793876c5abdd5931f922db,baby,38,38,38907.32,1023.88,1404.63
5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,63,63,37733.9,598.95,3991.91
53b36df67ebb7c41585e8d54d6772e08,watches_gifts,321,304,37454.63,116.68,2258.86
aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,520,425,37104.3,71.35,7093.45


In [0]:

# GOLD TABLE 4: DELIVERY ANALYTICS
# STEP 1: LOAD ORDERS DATA

from pyspark.sql.functions import (
    col, count, avg, sum,
    when, datediff, round
)

orders = spark.table(
    "workspace.silver.orders_clean"
)

print("ORDERS DATA LOADED SUCCESSFULLY")

print("Total records:", orders.count())

orders.printSchema()

display(orders.limit(10))


ORDERS DATA LOADED SUCCESSFULLY
Total records: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,source_file
2807d0e504d6d4894d41672727bc139f,72ae281627a6102d9b3718528b420f8a,delivered,2018-02-03T20:37:35.000Z,2018-02-03T20:50:22.000Z,2018-02-05T22:37:28.000Z,2018-02-08T16:13:46.000Z,2018-02-21T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv
ccbabeb0b02433bd0fcbac46e70339f2,c77ee2d8ba1614a4d489a44166894938,delivered,2018-02-19T20:31:09.000Z,2018-02-21T06:15:25.000Z,2018-02-22T21:04:23.000Z,2018-03-09T22:22:25.000Z,2018-03-13T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv
e4de6d53ecff736bc68804b0b6e9f635,9f6618c17568ac301465fe7ad056c674,delivered,2017-10-16T14:56:50.000Z,2017-10-17T03:49:34.000Z,2017-10-27T22:14:21.000Z,2017-11-08T21:25:24.000Z,2017-11-21T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv
cadbb3657dac2dbbd5b84b12e7b78aad,93ada7a24817edda9f4ab998fa823d16,delivered,2018-02-27T12:55:42.000Z,2018-03-01T02:48:54.000Z,2018-03-03T02:27:03.000Z,2018-03-16T14:59:01.000Z,2018-03-29T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv
d3d6788577c9592da441752e8a1dd5e3,8628fac2267e8c8804525da99c10ed0e,delivered,2017-09-19T22:17:15.000Z,2017-09-20T07:55:14.000Z,2017-09-22T17:23:09.000Z,2017-10-10T18:43:53.000Z,2017-10-13T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv
6a0a8bfbbe700284feb0845d95e0867f,68451b39b1314302c08c65a29f1140fc,delivered,2017-11-22T11:32:22.000Z,2017-11-22T11:46:50.000Z,2017-11-27T13:39:35.000Z,2017-12-28T19:43:00.000Z,2017-12-11T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv
0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03T17:44:42.000Z,2018-08-07T06:15:14.000Z,null,null,2018-08-21T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv
e1da8361c76cab67aa3588a1fbf1af54,dd854e24b40e3bc2b306946dee252015,delivered,2017-07-01T15:42:32.000Z,2017-07-04T04:10:19.000Z,2017-07-04T14:52:46.000Z,2017-07-12T20:17:57.000Z,2017-07-25T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv
4b3a605942f29d490cb74bd6ace6b9f0,e8a332c3433fbd37920ec89fc295d075,delivered,2017-09-29T09:32:17.000Z,2017-09-29T09:44:12.000Z,2017-09-29T21:58:48.000Z,2017-09-30T16:55:39.000Z,2017-10-17T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv
741fd1b80f015f170a7f458cf86769b8,df5c9e02596851fcbfc98804e4108bbd,delivered,2017-12-02T14:28:03.000Z,2017-12-02T14:38:30.000Z,2017-12-04T21:54:27.000Z,2017-12-11T16:17:04.000Z,2018-01-04T00:00:00.000Z,2026-11-16T06:16:32.377Z,olist_orders_dataset.csv


In [0]:

# GOLD TABLE 4: DELIVERY PERFORMANCE
# STEP 2: CALCULATE DELIVERY METRICS

from pyspark.sql.functions import (
    col, count, sum, avg,
    when, datediff, round
)

# Filter successfully delivered orders
delivery_df = orders.filter(
    (col("order_status") == "delivered") &
    col("order_purchase_timestamp").isNotNull() &
    col("order_delivered_customer_date").isNotNull() &
    col("order_estimated_delivery_date").isNotNull()
)

# Calculate delivery duration and delay
delivery_df = (
    delivery_df
    .withColumn(
        "delivery_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "estimated_days",
        datediff(
            col("order_estimated_delivery_date"),
            col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "delay_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_estimated_delivery_date")
        )
    )
)

# Create delivery analytics
delivery_analytics = delivery_df.agg(

    count("order_id").alias("total_delivered_orders"),

    round(avg("delivery_days"), 2)
    .alias("average_delivery_days"),

    round(avg("estimated_days"), 2)
    .alias("average_estimated_days"),

    sum(
        when(col("delay_days") > 0, 1).otherwise(0)
    ).alias("late_deliveries"),

    sum(
        when(col("delay_days") <= 0, 1).otherwise(0)
    ).alias("on_time_deliveries"),

    round(
        avg(when(col("delay_days") > 0, col("delay_days"))), 2
    ).alias("average_delay_days")

)

# Calculate on-time delivery percentage
delivery_analytics = delivery_analytics.withColumn(
    "on_time_delivery_percentage",
    round(
        (col("on_time_deliveries") /
         col("total_delivered_orders")) * 100,
        2
    )
)

print("GOLD DELIVERY PERFORMANCE ANALYTICS")
print("-------------------------------------")

display(delivery_analytics)


GOLD DELIVERY PERFORMANCE ANALYTICS
-------------------------------------


total_delivered_orders,average_delivery_days,average_estimated_days,late_deliveries,on_time_deliveries,average_delay_days,on_time_delivery_percentage
96470,12.5,24.37,6534,89936,10.62,93.23


In [0]:

# Save Gold Delivery Analytics

(delivery_analytics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.gold_delivery_analytics")
)

print("Gold Delivery Analytics saved successfully!")

display(
    spark.table("workspace.gold.gold_delivery_analytics")
)


Gold Delivery Analytics saved successfully!


total_delivered_orders,average_delivery_days,average_estimated_days,late_deliveries,on_time_deliveries,average_delay_days,on_time_delivery_percentage
96470,12.5,24.37,6534,89936,10.62,93.23


In [0]:

from pyspark.sql.functions import col, count, avg, sum, when, round

# Load Silver Reviews
reviews = spark.table("workspace.silver.reviews_clean")

# Calculate review analytics
review_analytics = reviews.agg(
    count("*").alias("total_reviews"),
    round(avg("review_score"), 2).alias("average_rating"),

    sum(when(col("review_score") == 1, 1).otherwise(0)).alias("one_star"),
    sum(when(col("review_score") == 2, 1).otherwise(0)).alias("two_star"),
    sum(when(col("review_score") == 3, 1).otherwise(0)).alias("three_star"),
    sum(when(col("review_score") == 4, 1).otherwise(0)).alias("four_star"),
    sum(when(col("review_score") == 5, 1).otherwise(0)).alias("five_star"),

    sum(when(col("review_score") >= 4, 1).otherwise(0)).alias("positive_reviews"),
    sum(when(col("review_score") == 3, 1).otherwise(0)).alias("neutral_reviews"),
    sum(when(col("review_score") <= 2, 1).otherwise(0)).alias("negative_reviews")
)

print("GOLD REVIEW ANALYTICS")
display(review_analytics)


GOLD REVIEW ANALYTICS


total_reviews,average_rating,one_star,two_star,three_star,four_star,five_star,positive_reviews,neutral_reviews,negative_reviews
99224,4.09,11424,3151,8179,19142,57328,76470,8179,14575


In [0]:

# Save Gold Review Analytics

(review_analytics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.gold_review_analytics")
)

print("Gold Review Analytics saved successfully!")

# Verify saved table
display(
    spark.table("workspace.gold.gold_review_analytics")
)


Gold Review Analytics saved successfully!


total_reviews,average_rating,one_star,two_star,three_star,four_star,five_star,positive_reviews,neutral_reviews,negative_reviews
99224,4.09,11424,3151,8179,19142,57328,76470,8179,14575


In [0]:

# GOLD TABLE: GEOGRAPHY ANALYTICS
# STEP 1: LOAD SILVER TABLES

customers = spark.table("workspace.silver.customers_clean")
orders = spark.table("workspace.silver.orders_clean")
order_items = spark.table("workspace.silver.order_items_clean")

print("GEOGRAPHY DATA LOADED SUCCESSFULLY")
print("-----------------------------------")

print("Total Customers:", customers.count())
print("Total Orders:", orders.count())
print("Total Order Items:", order_items.count())

print("Customer Columns:", customers.columns)
print("Order Columns:", orders.columns)
print("Order Item Columns:", order_items.columns)



GEOGRAPHY DATA LOADED SUCCESSFULLY
-----------------------------------
Total Customers: 99441
Total Orders: 99441
Total Order Items: 112650
Customer Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'ingestion_timestamp', 'source_file']
Order Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'ingestion_timestamp', 'source_file']
Order Item Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'ingestion_timestamp', 'source_file']


In [0]:

from pyspark.sql.functions import (
    col, countDistinct, sum, avg, round
)

# Join customers with orders
geo_orders = customers.join(
    orders,
    on="customer_id",
    how="inner"
)

# Keep delivered orders
geo_orders = geo_orders.filter(
    col("order_status") == "delivered"
)

# Join with order items
geo_sales = geo_orders.join(
    order_items,
    on="order_id",
    how="inner"
)

# Create city and state-level analytics
geography_analytics = (
    geo_sales
    .groupBy(
        "customer_state",
        "customer_city"
    )
    .agg(
        countDistinct("customer_unique_id")
        .alias("total_customers"),

        countDistinct("order_id")
        .alias("total_orders"),

        round(sum("price"), 2)
        .alias("total_revenue"),

        round(sum("freight_value"), 2)
        .alias("total_freight"),

        round(avg("price"), 2)
        .alias("average_item_price")
    )
)

print("GOLD GEOGRAPHY ANALYTICS CREATED")
print("--------------------------------")

print("Total Cities Analyzed:",
      geography_analytics.count())

display(
    geography_analytics
    .orderBy(col("total_revenue").desc())
    .limit(20)
)


GOLD GEOGRAPHY ANALYTICS CREATED
--------------------------------
Total Cities Analyzed: 4272


customer_state,customer_city,total_customers,total_orders,total_revenue,total_freight,average_item_price
SP,sao paulo,14528,15045,1859556.83,248403.34,106.87
RJ,rio de janeiro,6361,6601,955573.97,156158.24,125.87
MG,belo horizonte,2606,2697,346039.04,59911.47,112.1
DF,brasilia,2013,2071,295814.72,49384.33,126.36
PR,curitiba,1434,1489,205999.33,32460.39,119.28
SP,campinas,1363,1406,184657.87,24345.03,113.57
RS,porto alegre,1292,1342,182319.55,32486.29,115.91
BA,salvador,1154,1188,173273.05,34440.25,127.59
SP,guarulhos,1111,1144,139008.19,18727.46,107.43
RJ,niteroi,788,825,115890.26,19557.7,120.97


In [0]:

# Save Gold Geography Analytics
# Fix: Enable schema overwrite

(geography_analytics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_geography_analytics")
)

print("Gold Geography Analytics saved successfully!")

# Verify saved table
geo_gold = spark.table(
    "workspace.gold.gold_geography_analytics"
)

print("Total Cities:", geo_gold.count())

display(geo_gold.orderBy("total_revenue", ascending=False).limit(10))


Gold Geography Analytics saved successfully!
Total Cities: 4272


customer_state,customer_city,total_customers,total_orders,total_revenue,total_freight,average_item_price
SP,sao paulo,14528,15045,1859556.83,248403.34,106.87
RJ,rio de janeiro,6361,6601,955573.97,156158.24,125.87
MG,belo horizonte,2606,2697,346039.04,59911.47,112.1
DF,brasilia,2013,2071,295814.72,49384.33,126.36
PR,curitiba,1434,1489,205999.33,32460.39,119.28
SP,campinas,1363,1406,184657.87,24345.03,113.57
RS,porto alegre,1292,1342,182319.55,32486.29,115.91
BA,salvador,1154,1188,173273.05,34440.25,127.59
SP,guarulhos,1111,1144,139008.19,18727.46,107.43
RJ,niteroi,788,825,115890.26,19557.7,120.97


In [0]:

# FINAL GOLD LAYER VALIDATION

gold_tables = [
    "gold_customer_analytics",
    "gold_sales_analytics",
    "gold_product_analytics",
    "gold_delivery_analytics",
    "gold_review_analytics",
    "gold_geography_analytics"
]

print("FINAL GOLD LAYER VALIDATION")

for table in gold_tables:

    df = spark.table("workspace.gold." + table)

    print(
        table,
        "| Records:", df.count(),
        "| Columns:", len(df.columns)
    )

print("============================")
print("GOLD LAYER VALIDATION COMPLETED!")


FINAL GOLD LAYER VALIDATION
gold_customer_analytics | Records: 93358 | Columns: 9
gold_sales_analytics | Records: 25 | Columns: 6
gold_product_analytics | Records: 32216 | Columns: 7
gold_delivery_analytics | Records: 1 | Columns: 7
gold_review_analytics | Records: 1 | Columns: 10
gold_geography_analytics | Records: 4272 | Columns: 7
GOLD LAYER VALIDATION COMPLETED!
